In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

In [7]:
loader = PyPDFLoader("../data/a.pdf")
docs = loader.load()

In [8]:
len(docs)

15

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
splitted_data = splitter.split_documents(documents=docs)

In [10]:
len(splitted_data)

64

In [11]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3185.12it/s]


In [12]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [13]:
query = "What is self-attention?"
data = vector_store.similarity_search(query=query)


In [14]:
len(data)

4

In [15]:
data[0].page_content

'the number of operations required to relate signals from two arbitrary input or output positions grows\nin the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes\nit more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been'

In [16]:
context = ""
for doc in data:
    context+= doc.page_content + "\n"
print(context)

the number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,


In [17]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [18]:
res = llm.invoke(f"Can you provide me answer based on provided context for my question , context :{context} ans question : {query}")

In [19]:
print(res.content)

Based on the provided context, self-attention is an attention mechanism that relates different positions of a single sequence in order to compute a representation of the sequence. It allows the model to attend to different parts of the input sequence and weigh their importance, enabling it to capture long-range dependencies and contextual relationships within the sequence. In other words, self-attention is a way for the model to focus on specific parts of the input sequence when generating a representation of the sequence, rather than relying solely on recurrent or convolutional layers.


### Chain - Context generate | prompt | llm |StrOutputParser

In [22]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content +"\n"
    return {
        "context":context,
        "question":query
    }

In [23]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("""
    You are a helpful assistant , provide answer based on the context for the user question.
    If you don't know answer you can say "I don't know.
    Context : {context}
    question: {question}

""")

In [30]:
from langchain_core.output_parsers import StrOutputParser
out = StrOutputParser()

In [31]:
rag_chain = get_context|prompt|llm|out

In [32]:
res=rag_chain.invoke("What architecture does the Transformer avoid?")

In [35]:
res

'The Transformer avoids architectures that have a distance-based increase in the number of operations required to relate signals from two arbitrary input or output positions, specifically mentioning ConvS2S and ByteNet, which have linear and logarithmic increases respectively. Instead, the Transformer uses self-attention, which reduces this to a constant number of operations.'